这个代码2025年10月重启EPFLUX计算的测试代码，只使用ERA5数据进行计算看计算程序是否正确。

In [ ]:
# ============================================================
# Block A — Compute and cache EP-flux (all waves, do_ubar=True)
# ============================================================

import os, glob
import numpy as np
import xarray as xr
import aostools_functions as ac

out_root = "/home/weiji/restart_exam/plots/epflux_speedtest/WACCMnew20251019"
os.makedirs(out_root, exist_ok=True)

scenarios = [
    ("0008_Feb_couple",
     "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
     "BWCN.e122.f19_g16.002_0008/Feb/BWCN.e122.f19_g16.002.0008-02.*.nc"),
    ("0008_Mar_couple",
     "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
     "BWCN.e122.f19_g16.002_0008/Mar/BWCN.e122.f19_g16.002.0008-03.*.nc"),
    ("0008_Feb_nocouple",
     "/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc"),
    ("0008_Mar_nocouple",
     "/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc"),
    ("0008_long_3.1_6.1",
     "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
     "BWCN.e122.f19_g16.002/BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc")
]

for tag, pattern in scenarios:
    print(f"\n=== Processing {tag} ===")

    # 读取
    if "long" in tag:
        ds = xr.open_dataset(pattern, mask_and_scale=True).sel(
            time=slice("0008-03-01", "0008-06-01")
        )
    else:
        files = sorted(glob.glob(pattern))
        ds_list = [xr.open_dataset(fp, mask_and_scale=True) for fp in files]
        ds = xr.concat(ds_list, dim="time")

    # 屏蔽异常值
    for var in ("U", "V", "T"):
        ds[var] = ds[var].where(np.abs(ds[var]) < 1e20)

    # ---- 压力裁剪（两步，先Pa后hPa；hPa用 where 而不是 sel）----
    # Step 1: 用 Pa 裁 100–1 hPa 的大区间（10000→100 Pa）
    ds = ds.sortby("plev", ascending=False).sel(plev=slice(10000.0, 100.0))

    # Step 2: 用 hPa 再保守裁掉顶端（避免 1 hPa 非局地影响）
    top_hpa = 1.0   # 如需到 1 hPa，把 2.0 改为 1.0
    ds = ds.assign_coords(plev_hpa=(ds["plev"] / 100.0))
    ds = ds.where((ds.plev_hpa <= 100.0) & (ds.plev_hpa >= top_hpa), drop=True)

    # 提取变量（ComputeEPfluxDiv 一律用 hPa 传入）
    lat       = ds["lat"].values
    plev_hpa  = ds["plev_hpa"].values.astype(float)   # ✅ 用 hPa
    u = ds["U"].values
    v = ds["V"].values
    t = ds["T"].values

    print(f"  Data shape: {u.shape}, plev_hpa range: [{plev_hpa.min():.3f}, {plev_hpa.max():.3f}] hPa")

    # 保底：至少 3 层更稳（np.gradient至少需要2层；保留3层更安全）
    if plev_hpa.size < 3:
        raise ValueError("Not enough pressure levels after subsetting (need >= 3).")

    # 计算（全波，do_ubar=True）
    ep1, ep2, div1, div2 = ac.ComputeEPfluxDiv(
        lat, plev_hpa, u, v, t, wave=-1, do_ubar=True
    )

    # 时间平均
    ep1_mean = np.nanmean(ep1, axis=0)            # (plev, lat)
    ep2_mean = np.nanmean(ep2, axis=0)
    div_mean = np.nanmean(div1 + div2, axis=0)

    # 缓存
    cache_mean = os.path.join(out_root, f"epflux_mean_allwaves_{tag}.npz")
    cache_div  = os.path.join(out_root, f"epflux_div_mean_allwaves_{tag}.npz")
    np.savez_compressed(cache_mean, lat=lat, plev=plev_hpa, ep1=ep1_mean, ep2=ep2_mean)
    np.savez_compressed(cache_div,  lat=lat, plev=plev_hpa, div=div_mean)
    print(f"  ✅ Saved: {cache_mean}")
    print(f"  ✅ Saved: {cache_div}")

print("\nAll WACCM scenarios processed and cached!")


In [ ]:
# ============================================================
# Block B — Plot EP-flux vectors + divergence (ERA5-style)
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import aostools_functions as ac

# -------------------------
# 参数设置（可自由调整）
# -------------------------
root_dir    = "/home/weiji/restart_exam/plots/epflux_speedtest/WACCMnew20251019"
scenarios   = [
    "0008_Feb_couple",
    "0008_Mar_couple",
    "0008_Feb_nocouple",
    "0008_Mar_nocouple",
    "0008_long_3.1_6.1",
]
step_lat    = 1        # 箭头稀疏度（纬向）
step_plev   = 1         # 箭头稀疏度（垂直）
arrow_scale = 2e16      # 箭头缩放
levels_div  = np.linspace(-4, 4, 17)

# -------------------------
# 绘图循环
# -------------------------
for tag in scenarios:
    print(f"\n--- Plotting {tag} ---")

    mean_file = os.path.join(root_dir, f"epflux_mean_allwaves_{tag}.npz")
    div_file  = os.path.join(root_dir, f"epflux_div_mean_allwaves_{tag}.npz")
    if not (os.path.exists(mean_file) and os.path.exists(div_file)):
        print(f"⚠️  Missing cache for {tag}, skip.")
        continue

    V = np.load(mean_file)
    D = np.load(div_file)

    lat  = V["lat"];   plev = V["plev"]
    F1   = V["ep1"];   F2   = V["ep2"]
    DIV  = D["div"]


    plev_plot = plev
    F1_plot = F1
    F2_plot = F2
    DIV_plot = DIV

    # 散度底图
    fig, ax = plt.subplots(figsize=(8, 6))
    cf = ax.contourf(lat, plev_plot, DIV_plot,
                     levels=levels_div, cmap="RdBu_r", extend="both")

    # 转为 xarray 以便 PlotEPfluxArrows
    f1 = xr.DataArray(F1_plot, coords={"plev": plev_plot, "lat": lat},
                      dims=["plev", "lat"]).transpose("lat", "plev")
    f2 = xr.DataArray(F2_plot, coords={"plev": plev_plot, "lat": lat},
                      dims=["plev", "lat"]).transpose("lat", "plev")

    # 下采样箭头
    f1s = f1.isel(lat=slice(None, None, step_lat),
                  plev=slice(None, None, step_plev))
    f2s = f2.isel(lat=slice(None, None, step_lat),
                  plev=slice(None, None, step_plev))

    # 绘制 EP-flux 矢量
    ac.PlotEPfluxArrows(
        f1s.lat, f1s.plev, f1s, f2s,
        fig, ax,
        xlim=[-90, 90],
        ylim=[100, 1],
        yscale="log",
        scale=arrow_scale
    )

    # 外观统一
    ax.set_xlim([-90, 90])
    ax.set_ylim([100, 1])
    ax.set_yscale("log")
    ax.set_xlabel("Latitude (°)")
    ax.set_ylabel("Pressure (hPa)")
    ax.set_title(f"WACCM {tag}  |  EP-flux vectors over divergence (all waves)")

    cbar = fig.colorbar(cf, ax=ax, pad=0.02)
    cbar.set_label("EP-flux divergence [m/s/day]")

    # 保存图像
    outfile = os.path.join(root_dir, f"epflux_combined_allwaves_{tag}.png")
    fig.savefig(outfile, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  ✅ Saved: {outfile}")

print("\nAll plots done!")
